# 🛡️ PhishAlert — Evidencia 3

Detección de mensajes de phishing mediante procesamiento de lenguaje natural.

**Integrantes:** Marycielo Bedoya · María Pinazo · Patrick Ramírez

**Técnicas de NLP:** clasificación de textos (DistilBERT multilingüe afinado) y extracción de entidades (enlaces, correos, teléfonos, organizaciones).

Esta notebook clona el repositorio del proyecto y ejecuta todo el flujo: pruebas → datos → entrenamiento → demo web.

> Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**.

## 1. Preparación del entorno

In [ ]:
import torch
print('GPU disponible:', torch.cuda.is_available(), '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU (será lento)')

In [ ]:
REPO_URL = 'https://github.com/patrickram99/phishalert.git'

import os
%cd /content
if not os.path.exists('phishalert'):
    !git clone -q {REPO_URL} phishalert
%cd /content/phishalert
!git pull -q
!git log --oneline -3

In [ ]:
!pip install -q -r requirements.txt
!python -m spacy download xx_ent_wiki_sm -q

## 2. Pruebas automatizadas

Pruebas de la extracción de entidades, el resaltado y el pipeline (no requieren modelo).

In [ ]:
!python -m pytest -q

## 3. Datos

Descarga los datasets públicos en inglés, traduce una muestra balanceada al español, agrega el conjunto semilla en español y genera `data/phishalert.csv` con particiones train/val/test (spec 01).

In [ ]:
!python scripts/get_data.py --translate-n 3000

In [ ]:
import pandas as pd
df = pd.read_csv('data/phishalert.csv')
display(pd.crosstab([df['lang'], df['label']], df['split'], margins=True))
pd.set_option('display.max_colwidth', 160)
df[df['lang'] == 'es'].sample(6, random_state=1)[['text', 'label', 'source']]

## 4. Entrenamiento del clasificador

Afinado corto de `distilbert-base-multilingual-cased` (submuestra de 6000 mensajes, 1 época) para la demo al 50 % (spec 02). El entrenamiento completo corresponde a la Evidencia 4.

In [ ]:
!python scripts/train.py --epochs 1 --max-train 6000 --out models/phishalert

## 5. Prueba rápida: clasificación + entidades + resaltado

In [ ]:
import sys
sys.path.insert(0, '/content/phishalert/src')
from phishalert.classifier import PhishClassifier
from phishalert.pipeline import analyze
from app import EJEMPLOS

clf = PhishClassifier('models/phishalert')
for (texto,) in EJEMPLOS:
    r = analyze(texto, clf)
    print(f"[{r['label'].upper():9}] p(phishing)={r['prob_phishing']:.3f}  {texto[:70]}…")
    print('   entidades :', [(e['type'], e['text']) for e in r['entities']])
    print('   resaltados:', [(h['category'], h['text']) for h in r['highlights']])
    print()

## 6. Demo web

Lanza la aplicación Gradio con un enlace público temporal (spec 04).

In [ ]:
from app import build_demo

demo = build_demo('models/phishalert')
demo.launch(share=True)